# Type IV COPV Dome Optimization

This notebook demonstrates the end-to-end `FPP-JAX-Optimizer` workflow for a Type IV hydrogen tank dome:

1. Parameterize the oblate ellipsoidal dome geometry.
2. Evaluate the baseline helical spillover state.
3. Run differentiable optimization of FPP patch centers, orientations, sizes, stack intensity, and the hybrid winding transition boundary.
4. Inspect the structural, wrinkling, and thickness-gradient maps.
5. Export a Nastran `.bdf` deck and a summary JSON for downstream simulation workflows.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
from IPython.display import display

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from fpp_jax_optimizer import (
    DomeConfig,
    ExportConfig,
    MaterialConfig,
    OptimizationConfig,
    boss_opening_curve,
    build_type_iv_dome,
    evaluate_baseline_layout,
    optimize_patch_layout,
    surface_xyz,
    summarize_result,
    write_nastran_bdf,
    write_summary_json,
)


def build_plotly_layout(result):
    dome = result["dome"]
    optimized = result["optimized"]
    xyz = np.asarray(dome.xyz, dtype=float)
    total_plies = np.asarray(optimized["thickness"]["total_plies"], dtype=float)
    centers = np.asarray(
        surface_xyz(
            dome.config,
            optimized["layout"]["center_theta"],
            optimized["layout"]["center_phi"],
        ),
        dtype=float,
    )
    phi = np.linspace(0.0, 2.0 * np.pi, dome.phi.shape[0], endpoint=False)
    transition_theta = float(optimized["layout"]["transition_theta"])
    transition_curve = np.asarray(
        surface_xyz(dome.config, np.full_like(phi, transition_theta), phi),
        dtype=float,
    )
    boss_curve = boss_opening_curve(dome)

    fig = go.Figure()
    fig.add_trace(
        go.Surface(
            x=xyz[:, :, 0],
            y=xyz[:, :, 1],
            z=xyz[:, :, 2],
            surfacecolor=total_plies,
            colorscale="Viridis",
            colorbar_title="Total plies",
            opacity=0.92,
            showscale=True,
        )
    )
    fig.add_trace(
        go.Scatter3d(
            x=boss_curve[:, 0],
            y=boss_curve[:, 1],
            z=boss_curve[:, 2],
            mode="lines",
            line=dict(color="#ff6f61", width=6),
            name="Polar opening",
        )
    )
    fig.add_trace(
        go.Scatter3d(
            x=transition_curve[:, 0],
            y=transition_curve[:, 1],
            z=transition_curve[:, 2],
            mode="lines",
            line=dict(color="#111111", width=5, dash="dash"),
            name="Z-transition",
        )
    )
    fig.add_trace(
        go.Scatter3d(
            x=centers[:, 0],
            y=centers[:, 1],
            z=centers[:, 2],
            mode="markers+text",
            text=[f"Patch {idx + 1}" for idx in range(centers.shape[0])],
            textposition="top center",
            marker=dict(size=5, color="#ffd54f"),
            name="Patch centers",
        )
    )
    fig.update_layout(
        title="Optimized Type IV dome layout",
        scene=dict(aspectmode="data"),
        margin=dict(l=0, r=0, t=40, b=0),
    )
    return fig


def plot_maps(result):
    optimized = result["optimized"]
    dome = result["dome"]
    theta_deg = np.degrees(np.asarray(dome.theta, dtype=float))
    phi_deg = np.degrees(np.asarray(dome.phi, dtype=float))
    fields = [
        (np.asarray(optimized["structure"]["stress_index"], dtype=float), "Stress index", "inferno"),
        (100.0 * np.asarray(optimized["kinematics"]["shear_map"], dtype=float), "Shear strain [%]", "magma"),
        (np.asarray(optimized["thickness"]["thickness_gradient_mm_per_m"], dtype=float), "Thickness gradient [mm/m]", "viridis"),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout=True)
    for ax, (field, title, cmap) in zip(axes, fields):
        image = ax.imshow(
            field,
            origin="lower",
            aspect="auto",
            extent=[phi_deg[0], phi_deg[-1], theta_deg[0], theta_deg[-1]],
            cmap=cmap,
        )
        ax.set_title(title)
        ax.set_xlabel("phi [deg]")
        ax.set_ylabel("theta [deg]")
        plt.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    plt.show()


def plot_convergence(history):
    steps = [entry["step"] for entry in history]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(steps, [entry["loss"] for entry in history], label="Total loss", linewidth=2.0)
    ax.plot(steps, [entry["structural_loss"] for entry in history], label="Structural", linewidth=1.8)
    ax.plot(steps, [entry["kinematic_penalty"] for entry in history], label="Wrinkling penalty", linewidth=1.8)
    ax.plot(steps, [entry["thickness_penalty"] for entry in history], label="Thickness penalty", linewidth=1.8)
    ax.set_title("Optimization convergence")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Objective value")
    ax.legend(loc="upper right")
    plt.show()


## 1. Build The Demonstration Geometry

The validation case is an oblate ellipsoidal Type IV dome with a `50 mm` polar opening and a major/minor ratio of `1.414`.

In [ ]:
dome_config = DomeConfig(theta_points=40, phi_points=72)
material_config = MaterialConfig()
optimization_config = OptimizationConfig(
    patch_count=8,
    steps=140,
    learning_rate=0.022,
    patch_length_bounds_m=(0.03, 0.08),
    patch_width_bounds_m=(0.025, 0.06),
    shear_weight=220.0,
    thickness_weight=100.0,
    mass_weight=0.04,
    boss_margin_theta=0.08,
)
export_config = ExportConfig(output_dir=repo_root / "outputs")

dome = build_type_iv_dome(dome_config)
geometry_summary = {
    "major_radius_m": round(dome.config.major_radius_m, 4),
    "minor_radius_m": round(dome.config.minor_radius_m, 4),
    "polar_opening_radius_m": dome.config.polar_opening_radius_m,
    "theta_open_deg": round(np.degrees(dome.config.theta_open), 3),
    "total_area_m2": round(dome.total_area_m2, 4),
}
geometry_summary


## 2. Evaluate The Baseline Helical Spillover State

This is the reference state with the helical spillover layer active and the FPP reinforcement effectively removed.

In [ ]:
baseline = evaluate_baseline_layout(dome=dome, material=material_config, config=optimization_config)
baseline_metrics = {
    "loss": round(float(baseline["metrics"]["loss"]), 4),
    "peak_stress_index": round(float(baseline["metrics"]["peak_stress_index"]), 4),
    "total_mass_kg": round(float(baseline["metrics"]["total_mass_kg"]), 4),
    "max_shear": round(float(baseline["metrics"]["max_shear"]), 5),
    "transition_height_m": round(float(baseline["metrics"]["transition_height_m"]), 4),
}
baseline_metrics


## 3. Run The Differentiable Optimization

Patch centers, angles, sizes, ply intensity, and the helical/FPP transition boundary are optimized together through analytical gradients.

In [ ]:
result = optimize_patch_layout(
    dome_config=dome_config,
    material=material_config,
    config=optimization_config,
)
summary = summarize_result(result)

optimization_summary = {
    "baseline_peak_stress_index": round(summary["baseline_peak_stress_index"], 4),
    "optimized_peak_stress_index": round(summary["optimized_peak_stress_index"], 4),
    "optimized_mass_kg": round(summary["optimized_mass_kg"], 4),
    "patch_mass_kg": round(summary["patch_mass_kg"], 4),
    "helical_mass_kg": round(summary["helical_mass_kg"], 4),
    "cost_savings_vs_all_fpp_pct": round(summary["cost_savings_vs_all_fpp_pct"], 2),
    "max_shear": round(summary["max_shear"], 5),
    "max_areal_distortion": round(summary["max_areal_distortion"], 5),
    "max_thickness_gradient_mm_per_m": round(summary["max_thickness_gradient_mm_per_m"], 4),
}
optimization_summary


In [ ]:
plot_convergence(result["history"])
plot_maps(result)


## 4. Inspect The Optimized 3D Layout

The interactive Plotly view shows the total ply buildup on the dome, patch centers, the polar opening, and the optimized winding/FPP transition boundary.

In [ ]:
layout_figure = build_plotly_layout(result)
layout_figure


## 5. Export The Dome To Downstream FEA

The final section writes a `.bdf` deck plus a JSON summary and an interactive HTML layout for handoff and review.

In [ ]:
output_dir = export_config.output_dir
output_dir.mkdir(parents=True, exist_ok=True)

bdf_path = write_nastran_bdf(result, output_dir / export_config.bdf_filename, material=material_config)
summary_path = write_summary_json(result, output_dir / export_config.summary_filename)
html_path = output_dir / export_config.plotly_html_filename
layout_figure.write_html(html_path)

artifact_summary = {
    "bdf": str(bdf_path),
    "summary_json": str(summary_path),
    "plotly_html": str(html_path),
}
artifact_summary


In [ ]:
json.loads(summary_path.read_text(encoding="utf-8"))
